In [1]:
# ==========================================
# 实验：修正归一化方式（在fold内部计算min/max）
# 目的：消除数据泄露，验证结果是否变化
# ==========================================

import numpy as np
import pandas as pd
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import MinMaxScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("NGAFID 实验 - 修正归一化（fold内计算min/max）")
print("="*60)

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'

with open(os.path.join(data_dir, 'flight_data.pkl'), 'rb') as f:
    data = pickle.load(f)

header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))

print(f"\n✅ 数据加载成功")
print(f"  航班数: {len(data)}")
print(f"  header记录数: {len(header_df)}")

# ==========================================
# 2. 筛选论文 Section 3.2 的基准子集
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())

filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"\n📊 筛选后航班数: {len(filtered_header)}")
print(f"  维护后 (0): {sum(filtered_header['before_after']==0)}")
print(f"  维护前 (1): {sum(filtered_header['before_after']==1)}")

# ==========================================
# 3. 准备特征和标签（不预先归一化）
# ==========================================
target_len = 4096
X_list = []
y = []

print("\n⏳ 准备数据（不预先归一化）...")

for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 只截取，不归一化（将在fold内进行）
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"\n✅ 数据准备完成")
print(f"  X 形状: {X.shape}")
print(f"  标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 5折交叉验证（fold内归一化）
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

minirocket = MiniRocketMultivariate(
    random_state=42,
    num_kernels=10000,
)

classifier = LogisticRegressionCV(
    Cs=10, 
    cv=3, 
    random_state=42, 
    max_iter=5000
)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("5折交叉验证（fold内归一化）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx)} 样本 (0={sum(y_train==0)}, 1={sum(y_train==1)})")
    print(f"  验证集: {len(val_idx)} 样本 (0={sum(y_val==0)}, 1={sum(y_val==1)})")
    
    start = time.time()
    
    # 🔥 关键修正：在fold内拟合scaler
    # X shape: (n_samples, length, channels) -> 需要reshape成 (n_samples*length, channels)
    n_samples, length, channels = X_train.shape
    X_train_flat = X_train.reshape(-1, channels)
    scaler = MinMaxScaler()
    scaler.fit(X_train_flat)  # 只使用训练集拟合
    
    # 归一化训练集
    X_train_norm = scaler.transform(X_train_flat).reshape(n_samples, length, channels)
    
    # 归一化验证集
    X_val_flat = X_val.reshape(-1, channels)
    X_val_norm = scaler.transform(X_val_flat).reshape(X_val.shape)
    
    # MiniRocket 特征提取
    X_train_transform = minirocket.fit_transform(X_train_norm, y_train)
    X_val_transform = minirocket.transform(X_val_norm)
    
    # 训练分类器
    classifier.fit(X_train_transform, y_train)
    
    # 预测
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*60)
print("📊 修正归一化后的最终结果:")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)

# ==========================================
# 6. 对比
# ==========================================
print("\n📊 归一化方式对比:")
print("-" * 50)
print(f"  全局归一化 (stats.csv):  ~54.3%")
print(f"  fold内归一化 (修正后):   {np.mean(accuracies):.4f}")
print("-" * 50)

NGAFID 实验 - 修正归一化（fold内计算min/max）

✅ 数据加载成功
  航班数: 11446
  header记录数: 11446

📊 筛选后航班数: 11446
  维护后 (0): 5844
  维护前 (1): 5602

⏳ 准备数据（不预先归一化）...

✅ 数据准备完成
  X 形状: (11446, 4096, 23)
  标签分布: 0=5844, 1=5602

5折交叉验证（fold内归一化）

--- Fold 1 ---
  训练集: 9156 样本 (0=4675, 1=4481)
  验证集: 2290 样本 (0=1169, 1=1121)
  准确率: 0.5786, F1: 0.5663, AUC: 0.6149, 耗时: 147.49s

--- Fold 2 ---
  训练集: 9157 样本 (0=4676, 1=4481)
  验证集: 2289 样本 (0=1168, 1=1121)
  准确率: 0.6073, F1: 0.5960, AUC: 0.6409, 耗时: 141.77s

--- Fold 3 ---
  训练集: 9157 样本 (0=4675, 1=4482)
  验证集: 2289 样本 (0=1169, 1=1120)
  准确率: 0.5745, F1: 0.5593, AUC: 0.6124, 耗时: 154.54s

--- Fold 4 ---
  训练集: 9157 样本 (0=4675, 1=4482)
  验证集: 2289 样本 (0=1169, 1=1120)
  准确率: 0.5950, F1: 0.5757, AUC: 0.6406, 耗时: 137.50s

--- Fold 5 ---
  训练集: 9157 样本 (0=4675, 1=4482)
  验证集: 2289 样本 (0=1169, 1=1120)
  准确率: 0.5810, F1: 0.5607, AUC: 0.6151, 耗时: 142.10s

📊 修正归一化后的最终结果:
  准确率: 0.5873 ± 0.0121
  F1分数: 0.5716 ± 0.0135
  AUC:    0.6248 ± 0.0131

📊 归一化方式对比:
------------------